In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('Iris.csv')

print('=' * 70)
print('VERI SETI BOYUTU')
print('=' * 70)
print(f'Satir sayisi: {df.shape[0]}')
print(f'Sutun sayisi: {df.shape[1]}')
print()

print('=' * 70)
print('SUTUNLAR VE VERI TIPLERI')
print('=' * 70)
for col in df.columns:
    print(f"{'%-25s' % col} | dtype: {str(df[col].dtype):<15} | Benzersiz: {df[col].nunique()}")

VERI SETI BOYUTU
Satir sayisi: 150
Sutun sayisi: 6

SUTUNLAR VE VERI TIPLERI
Id                        | dtype: int64           | Benzersiz: 150
SepalLengthCm             | dtype: float64         | Benzersiz: 35
SepalWidthCm              | dtype: float64         | Benzersiz: 23
PetalLengthCm             | dtype: float64         | Benzersiz: 43
PetalWidthCm              | dtype: float64         | Benzersiz: 22
Species                   | dtype: object          | Benzersiz: 3


In [3]:
print('=' * 70)
print('DEGISKEN ISIMLERI VE ACILAMALARI')
print('=' * 70)

variable_info = {
    'Id': 'Her gozlem icin benzersiz kimlik numarasi',
    'SepalLengthCm': 'Canak yapragi uzunlugu (cm)',
    'SepalWidthCm': 'Canak yapragi genisligi (cm)',
    'PetalLengthCm': 'Tac yapragi uzunlugu (cm)',
    'PetalWidthCm': 'Tac yapragi genisligi (cm)',
    'Species': 'Cicek turu (HEDEF DEGISKEN)'
}

for var, desc in variable_info.items():
    print(f"{'%-22s' % var} | {desc}")

print()


print('=' * 70)
print('HEDEF DEGISKEN: Species')
print('=' * 70)

print(df['Species'].value_counts().to_string())

DEGISKEN ISIMLERI VE ACILAMALARI
Id                     | Her gozlem icin benzersiz kimlik numarasi
SepalLengthCm          | Canak yapragi uzunlugu (cm)
SepalWidthCm           | Canak yapragi genisligi (cm)
PetalLengthCm          | Tac yapragi uzunlugu (cm)
PetalWidthCm           | Tac yapragi genisligi (cm)
Species                | Cicek turu (HEDEF DEGISKEN)

HEDEF DEGISKEN: Species
Species
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50


In [4]:
print('=' * 70)
print('HEDEF DEGISKENIN DAGILIMI')
print('=' * 70)

cat_columns = ['Species']

for col in cat_columns:
    print(f'\n{"-" * 50}')
    print(f'{col.upper()}')
    print(f'{"-" * 50}')

    for val, cnt in df[col].value_counts().items():
        print(f"  {'%-25s' % str(val)} : {cnt:5d} ({cnt/len(df)*100:5.2f}%)")

HEDEF DEGISKENIN DAGILIMI

--------------------------------------------------
SPECIES
--------------------------------------------------
  Iris-setosa               :    50 (33.33%)
  Iris-versicolor           :    50 (33.33%)
  Iris-virginica            :    50 (33.33%)


In [5]:
print('=' * 70)
print('SAYISAL DEGISKENLERIN ISTATISTIKLERI')
print('=' * 70)

num_cols = [
    'SepalLengthCm',
    'SepalWidthCm',
    'PetalLengthCm',
    'PetalWidthCm'
]

print(df[num_cols].describe().to_string())

SAYISAL DEGISKENLERIN ISTATISTIKLERI
       SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm
count     150.000000    150.000000     150.000000    150.000000
mean        5.843333      3.054000       3.758667      1.198667
std         0.828066      0.433594       1.764420      0.763161
min         4.300000      2.000000       1.000000      0.100000
25%         5.100000      2.800000       1.600000      0.300000
50%         5.800000      3.000000       4.350000      1.300000
75%         6.400000      3.300000       5.100000      1.800000
max         7.900000      4.400000       6.900000      2.500000


In [6]:
#Bağımsız ve hedef değişkenleri belirliyoruz
feature_cols = [
    'SepalLengthCm',
    'SepalWidthCm',
    'PetalLengthCm',
    'PetalWidthCm'
]

X = df[feature_cols].copy()
y = df['Species']


print('KULLANILACAK BAGIMSIZ DEGISKENLER (X):')
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {col}")

print(f'\nHEDEF DEGISKEN (y): Species')

KULLANILACAK BAGIMSIZ DEGISKENLER (X):
   1. SepalLengthCm
   2. SepalWidthCm
   3. PetalLengthCm
   4. PetalWidthCm

HEDEF DEGISKEN (y): Species


In [7]:

print('BAGIMSIZ DEGISKENLER (X) ZATEN SAYISAL')
print(f'\n{"Degisken":<25} {"Veri Tipi":<20}')
print('─' * 50)

for col in feature_cols:
    print(f"{col:<25} {str(X[col].dtype):<20}")


#Hedef değişkeni sayısal hale getiriyoruz
le_y = LabelEncoder()

y_encoded = le_y.fit_transform(y)

print(f'\nHedef degisken (Species) kodlama:')
for i, cls in enumerate(le_y.classes_):
    print(f"  {i} -> {cls}")

BAGIMSIZ DEGISKENLER (X) ZATEN SAYISAL

Degisken                  Veri Tipi           
──────────────────────────────────────────────────
SepalLengthCm             float64             
SepalWidthCm              float64             
PetalLengthCm             float64             
PetalWidthCm              float64             

Hedef degisken (Species) kodlama:
  0 -> Iris-setosa
  1 -> Iris-versicolor
  2 -> Iris-virginica


In [8]:
#Train / Test ayrımı
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print('VERI AYRIMI (Train/Test)')
print(f'  Egitim seti  : {X_train.shape[0]} ornek (%80)')
print(f'  Test seti    : {X_test.shape[0]} ornek (%20)')
print(f'  Stratifiy    : Evet (sinif dagilimi korundu)')

VERI AYRIMI (Train/Test)
  Egitim seti  : 120 ornek (%80)
  Test seti    : 30 ornek (%20)
  Stratifiy    : Evet (sinif dagilimi korundu)


In [9]:
#Veri ölçeklendirme
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('VERI OLCEKLENDIRILDI (StandardScaler)')
print(f'  Ortalama (her sutun icin): 0')
print(f'  Standart sapma (her sutun icin): 1')
print(f'  X_train_scaled boyutu: {X_train_scaled.shape}')
print(f'  X_test_scaled boyutu : {X_test_scaled.shape}')

VERI OLCEKLENDIRILDI (StandardScaler)
  Ortalama (her sutun icin): 0
  Standart sapma (her sutun icin): 1
  X_train_scaled boyutu: (120, 4)
  X_test_scaled boyutu : (30, 4)


**SVM MODELLERİ:**

In [10]:
print('=' * 50)
print('LINEAR SVM')
print('=' * 50)

svm_linear = SVC(
    kernel='linear',
    random_state=42
)

svm_linear.fit(X_train_scaled, y_train)

y_pred_linear = svm_linear.predict(X_test_scaled)

print(f'Test dogrulugu: {accuracy_score(y_test, y_pred_linear):.4f}')
print(f'Destek vektor sayisi: {svm_linear.n_support_.sum()}')

print('\nSiniflandirma Raporu:')
print(
    classification_report(
        y_test,
        y_pred_linear,
        target_names=le_y.classes_
    )
)

LINEAR SVM
Test dogrulugu: 1.0000
Destek vektor sayisi: 23

Siniflandirma Raporu:
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        10
Iris-versicolor       1.00      1.00      1.00        10
 Iris-virginica       1.00      1.00      1.00        10

       accuracy                           1.00        30
      macro avg       1.00      1.00      1.00        30
   weighted avg       1.00      1.00      1.00        30



In [11]:
print('=' * 50)
print('RBF SVM (Varsayilan)')
print('=' * 50)

svm_rbf = SVC(
    kernel='rbf',
    random_state=42
)

svm_rbf.fit(X_train_scaled, y_train)

y_pred_rbf = svm_rbf.predict(X_test_scaled)

print(f'Test dogrulugu: {accuracy_score(y_test, y_pred_rbf):.4f}')
print(f'Destek vektor sayisi: {svm_rbf.n_support_.sum()}')

print('\nSiniflandirma Raporu:')
print(
    classification_report(
        y_test,
        y_pred_rbf,
        target_names=le_y.classes_
    )
)

RBF SVM (Varsayilan)
Test dogrulugu: 0.9667
Destek vektor sayisi: 47

Siniflandirma Raporu:
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        10
Iris-versicolor       1.00      0.90      0.95        10
 Iris-virginica       0.91      1.00      0.95        10

       accuracy                           0.97        30
      macro avg       0.97      0.97      0.97        30
   weighted avg       0.97      0.97      0.97        30



In [12]:
print('=' * 50)
print('POLINOMIAL SVM (degree=3)')
print('=' * 50)

svm_poly = SVC(
    kernel='poly',
    degree=3,
    random_state=42
)

svm_poly.fit(X_train_scaled, y_train)

y_pred_poly = svm_poly.predict(X_test_scaled)

print(f'Test dogrulugu: {accuracy_score(y_test, y_pred_poly):.4f}')

POLINOMIAL SVM (degree=3)
Test dogrulugu: 0.9000


In [13]:
#GridSearchCV için:
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.1, 0.01],
    'kernel': ['rbf']
}

print('ARANAN PARAMETRELER:')

for k, v in param_grid.items():
    print(f'  {k}: {v}')

print(
    f'\nToplam kombinasyon: '
    f'{len(param_grid["C"]) * len(param_grid["gamma"])}'
)

print(
    f'Toplam model (5-fold CV ile): '
    f'{len(param_grid["C"]) * len(param_grid["gamma"]) * 5}'
)

ARANAN PARAMETRELER:
  C: [0.1, 1, 10, 100]
  gamma: ['scale', 'auto', 0.1, 0.01]
  kernel: ['rbf']

Toplam kombinasyon: 16
Toplam model (5-fold CV ile): 80


In [14]:
#GridSearchCV ile en iyi RBF SVM parametrelerini arıyoruz
grid_search = GridSearchCV(
    SVC(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)


print(f'\n{"=" * 50}')
print('EN IYI PARAMETRELER')
print('=' * 50)

print(f'C          : {grid_search.best_params_["C"]}')
print(f'gamma      : {grid_search.best_params_["gamma"]}')
print(f'kernel     : {grid_search.best_params_["kernel"]}')
print(f'CV skoru   : {grid_search.best_score_:.4f}')

Fitting 5 folds for each of 16 candidates, totalling 80 fits

EN IYI PARAMETRELER
C          : 1
gamma      : 0.1
kernel     : rbf
CV skoru   : 0.9833


In [15]:
#En iyi SVM modelini test ediyoruz
best_svm = grid_search.best_estimator_

y_pred_best = best_svm.predict(X_test_scaled)


print('=' * 50)
print('EN IYI MODEL SONUCLARI')
print('=' * 50)

print(f'Test dogrulugu: {accuracy_score(y_test, y_pred_best):.4f}')
print(f'Cross-validation skoru: {grid_search.best_score_:.4f}')

print('\nSiniflandirma Raporu:')
print(
    classification_report(
        y_test,
        y_pred_best,
        target_names=le_y.classes_
    )
)

EN IYI MODEL SONUCLARI
Test dogrulugu: 0.9667
Cross-validation skoru: 0.9833

Siniflandirma Raporu:
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        10
Iris-versicolor       1.00      0.90      0.95        10
 Iris-virginica       0.91      1.00      0.95        10

       accuracy                           0.97        30
      macro avg       0.97      0.97      0.97        30
   weighted avg       0.97      0.97      0.97        30



In [16]:
print('=' * 50)
print('KARISIKLIK MATRISI (Confusion Matrix)')
print('=' * 50)

cm = confusion_matrix(y_test, y_pred_best)

print('Satirlar: GERCEK deger | Sutunlar: TAHMIN edilen deger')
print()

header = ' ' * 22 + ' '.join(
    f'{c:>18}' for c in le_y.classes_
)

print(header)
print('─' * len(header))

for i, row in enumerate(cm):
    print(
        f"{le_y.classes_[i]:<22}" +
        ' '.join(f'{val:>18}' for val in row)
    )

print('Yorum: Kosegen (diagonal) ne kadar yuksekse model o kadar basarili.')

KARISIKLIK MATRISI (Confusion Matrix)
Satirlar: GERCEK deger | Sutunlar: TAHMIN edilen deger

                             Iris-setosa    Iris-versicolor     Iris-virginica
──────────────────────────────────────────────────────────────────────────────
Iris-setosa                           10                  0                  0
Iris-versicolor                        0                  9                  1
Iris-virginica                         0                  0                 10
Yorum: Kosegen (diagonal) ne kadar yuksekse model o kadar basarili.


In [17]:
print('=' * 50)
print('DESTEK VEKTOR ANALIZI')
print('=' * 50)

print(f'Toplam destek vektor sayisi : {best_svm.n_support_.sum()}')
print(f'Toplam egitim ornegi        : {len(X_train_scaled)}')
print(
    f'Destek vektor orani         : '
    f'{best_svm.n_support_.sum() / len(X_train_scaled) * 100:.1f}%'
)

print()
print('Her sinif icin destek vektor sayisi:')

for i, cls in enumerate(le_y.classes_):
    print(f'  {cls:<20} : {best_svm.n_support_[i]} vektor')


DESTEK VEKTOR ANALIZI
Toplam destek vektor sayisi : 50
Toplam egitim ornegi        : 120
Destek vektor orani         : 41.7%

Her sinif icin destek vektor sayisi:
  Iris-setosa          : 6 vektor
  Iris-versicolor      : 23 vektor
  Iris-virginica       : 21 vektor


In [18]:
print('=' * 50)
print('ORNEK TAHMIN (Test Setinden 10 Kayit)')
print('=' * 50)

print(f'{"No":<5} {"Tahmin":<22} {"Gercek":<22} {"Dogru?":<8}')
print('─' * 55)

for i in range(10):
    tahmin = le_y.inverse_transform(
        [best_svm.predict(X_test_scaled[i].reshape(1, -1))[0]]
    )[0]

    gercek = le_y.inverse_transform(
        [y_test[i]]
    )[0]

    dogru = '✓' if tahmin == gercek else '✗'

    print(f"{i+1:<5} {tahmin:<22} {gercek:<22} {dogru:<8}")


ORNEK TAHMIN (Test Setinden 10 Kayit)
No    Tahmin                 Gercek                 Dogru?  
───────────────────────────────────────────────────────
1     Iris-setosa            Iris-setosa            ✓       
2     Iris-virginica         Iris-virginica         ✓       
3     Iris-versicolor        Iris-versicolor        ✓       
4     Iris-versicolor        Iris-versicolor        ✓       
5     Iris-setosa            Iris-setosa            ✓       
6     Iris-versicolor        Iris-versicolor        ✓       
7     Iris-setosa            Iris-setosa            ✓       
8     Iris-setosa            Iris-setosa            ✓       
9     Iris-virginica         Iris-virginica         ✓       
10    Iris-versicolor        Iris-versicolor        ✓       
